In [1]:
from pathlib import Path

import automech as am
import polars as pl
from automech.reaction import Reaction
from project_utilities import p_

open_browser = True
tag = "Z_mess_v4"
root_path = Path("../..")

stoich1 = "C5H7O2"
stoich2 = "C5H9O2"

mech1_json = p_.calculated_pes_mechanism(tag, stoich1, "json", path=p_.data(root_path))
mech2_json = p_.calculated_pes_mechanism(tag, stoich2, "json", path=p_.data(root_path))
mech1 = am.io.read(mech1_json)
mech2 = am.io.read(mech2_json)

if open_browser:
    disp_mech1 = mech1.model_copy(deep=True)
    disp_mech2 = mech2.model_copy(deep=True)
    disp_mech1.reactions = disp_mech1.reactions.filter(~pl.col("well_skipping"))
    disp_mech2.reactions = disp_mech2.reactions.filter(~pl.col("well_skipping"))
    am.display(disp_mech1, out_name=f"{stoich1}.html")
    am.display(disp_mech2, out_name=f"{stoich2}.html")

In [7]:
import altair as alt
import numpy as np
from autochem.util import plot
from project_utilities import fig

P_range = (0.1, 100)
T = 700
T_range = (500, 1200)
P = 10
total = False
legend = True
min_branch_frac = 0.001
rate_df1a = fig.rate_data(
    mech1,
    reactants=("S(602)",),
    min_branch_frac=min_branch_frac,
)

rate_df1b = fig.rate_data(
    mech1,
    reactants=("S(1206)rR",),
    min_branch_frac=min_branch_frac,
)

rate_df2 = fig.rate_data(
    mech2,
    reactants=("S(719)",),
    min_branch_frac=min_branch_frac,
)

In [8]:
import polars as pl
from project_utilities.fig import RateData

T = 825
P = 1
rate_df = rate_df1b

objs = rate_df.get_column(RateData.branch_frac_obj).to_list()
prods = [tuple(obj.products) for obj in objs]
vals = [obj.rate(T=T, P=P).item() for obj in objs]

data = dict(zip(prods, vals, strict=True))
data

{('C5H7(500)', 'O2(6)'): 1.0}